<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Weather_LLM_wrapper_c12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1 Install Packages

In [34]:
!pip install openai requests

In [36]:
!pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.2 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0


In [17]:
import gradio
print(gradio.__version__)

6.31.0


### Step 2 Add OpenAI Keys

In [2]:
import os
from google.colab import userdata

import openai
# Retrieve API keys from Colab's secure storage
openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

### Step 3A Testing Weather App Database

Change parameters and try opening in website

http://api.openweathermap.org/data/2.5/weather?q=Dubai&appid=a7b109315b6ced6a9cd5177bdb98ca82&units=metric

In [6]:
#openweather get your own keys =https://home.openweathermap.org/api_keys
import requests

def get_current_weather(location, unit='celsius'):
    weather_api_key = "a7b109315b6ced6a9cd5177bdb98ca82"
    base_url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={weather_api_key}&units=metric"
    response = requests.get(base_url)
    data = response.json()

    weather_description = data['weather'][0]['description']

    return {
        "location": location,
        "temperature": data['main']['temp'],
        "weather": weather_description
    }

### Step 3B Raw JSON Output from OpenWeatherMap API

This cell demonstrates how to get and display the raw JSON response from the OpenWeatherMap API, which is the content of the `data` variable inside the `get_current_weather` function.

In [7]:
import json

weather_api_key = "a7b109315b6ced6a9cd5177bdb98ca82"
location = "Dubai"
base_url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={weather_api_key}&units=metric"
response = requests.get(base_url)
raw_json_data = response.json()

print(json.dumps(raw_json_data, indent=2))

{
  "coord": {
    "lon": 55.3047,
    "lat": 25.2582
  },
  "weather": [
    {
      "id": 802,
      "main": "Clouds",
      "description": "scattered clouds",
      "icon": "03d"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 29.96,
    "feels_like": 31.15,
    "temp_min": 29.14,
    "temp_max": 29.96,
    "pressure": 1017,
    "humidity": 51,
    "sea_level": 1017,
    "grnd_level": 1018
  },
  "visibility": 10000,
  "wind": {
    "speed": 3.6,
    "deg": 230
  },
  "clouds": {
    "all": 40
  },
  "dt": 1795247755,
  "sys": {
    "type": 1,
    "id": 7537,
    "country": "AE",
    "sunrise": 1795228801,
    "sunset": 1795267767
  },
  "timezone": 14400,
  "id": 292223,
  "name": "Dubai",
  "cod": 200
}


In [8]:
base_url

'http://api.openweathermap.org/data/2.5/weather?q=Dubai&appid=a7b109315b6ced6a9cd5177bdb98ca82&units=metric'

In [9]:
print(get_current_weather("Dubai"))

{'location': 'Dubai', 'temperature': 29.96, 'weather': 'scattered clouds'}


### Step 4 Create the Function Calling

In [10]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city, e.g. San Francisco"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["location"]
        }
    }
]

In [11]:
functions

[{'name': 'get_current_weather',
  'description': 'Get the current weather in a given location',
  'parameters': {'type': 'object',
   'properties': {'location': {'type': 'string',
     'description': 'The city, e.g. San Francisco'},
    'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}},
   'required': ['location']}}]

### Step 5 Creating Weather Chat LLM with funciton

In [2]:
def get_current_weather(location):
    # Demo data. Replace with a real weather API call if you want live data.
    return {"location": location, "temperature_c": 31, "condition": "Sunny", "humidity_percent": 55}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name, e.g. Dubai"}
                },
                "required": ["location"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [7]:
## Settings
import os
from google.colab import userdata

import openai
from openai import OpenAI
# Retrieve API keys from Colab's secure storage
openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


MODEL = "gpt-6-luna"
REASONING_EFFORT = "none"   # none = direct answers (generation-style); low/medium/high = thinking before answering

In [8]:
def weather_chat(user_message):
    messages = [
        {"role": "system", "content": "You are a weather bot. Use the get_current_weather tool for weather questions. For anything else, politely say you only provide weather updates."},
        {"role": "user", "content": user_message},
    ]

    # Step 1: send the message and the tool definition to the model
    response = client.chat.completions.create(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        messages=messages,
        tools=tools,
    )
    message = response.choices[0].message

    # No tool needed: return the model's direct answer
    if not message.tool_calls:
        return message.content

    # Step 2: record the model's tool request, run each tool, and add the results
    messages.append(message)
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        weather_data = get_current_weather(arguments["location"])
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(weather_data),
        })

    # Step 3: the LLM turns the raw data into a friendly answer
    final_response = client.chat.completions.create(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        messages=messages,
        tools=tools,
    )
    return final_response.choices[0].message.content

### Step 6 Testing the weather App

In [10]:
import json
import openai
openai.api_key = openai_api_key
weather_chat("what is temperature of dubai")

'Dubai is currently 31°C and sunny. Humidity is 55%.'

In [13]:
weather_chat("best pizza place in dubai")

'I only provide weather updates. Would you like the current weather in Dubai?'

In [11]:
#qc
weather_chat("كم درجة الحرارة في الشارقة؟ ")

'درجة الحرارة في الشارقة حاليًا 31°م، والجو مشمس. الرطوبة 55٪.'

### Step 7 Make this in Gradio

In [15]:
# Define Gradio interface
import gradio as gr
iface = gr.Interface(
    fn=weather_chat,
    inputs=gr.Textbox(label="Weather Queries"),
    outputs=gr.Textbox(label="Weather Updates"),
    title = "Weather Bot",
    description = "Ask me anything about weather!"
)

# Launch the Gradio interface
iface.launch(share="True")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f9503b31ea06b9be8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Step 8 Beautificaiton using AI

---
Use this prompt as example

So can you make this above creative application very professional? Make it 2-column layout, add blocks, add a logo, add center alignment, give a name uh, of, you can put your name like Niven Weatherbot, and put a 2-column, give the top cities of the world. Ca- Uh, You can create a dashboard with the current most important cities that are there, and put everything, put a live dashboard of the, or you can put weather trends that are there. Then create a chat, also create a chatbot button, or a tab that has the, you can chat and get the weather information. Use all the information above to get, or write all your code. You can use the help of open weather APIs to get all this information that are there. If you need something more, let do it as well. Make it look very professional, very, add proper layouts, add proper, uh, what else, add proper padding space, make central aligned, FAQs. Please make as professional as possible and share me the details. use the above logic and llm and related json using gradio 6.31.0 dont use copy buttion box


In [22]:
import gradio as gr
import requests
import json

# Helper function to get real live weather data for the Dashboard using the API key
def fetch_city_status(city_name):
    try:
        weather_api_key = "a7b109315b6ced6a9cd5177bdb98ca82"
        base_url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={weather_api_key}&units=metric"
        res = requests.get(base_url, timeout=5).json()
        temp = res['main']['temp']
        desc = res['weather'][0]['description'].title()
        hum = res['main']['humidity']
        return f"🌡️ {temp}°C | {desc} | 💧 {hum}%"
    except Exception:
        return "⚠️ Connection offline"

def get_all_dashboard_cities():
    return (
        fetch_city_status("Dubai"),
        fetch_city_status("London"),
        fetch_city_status("New York"),
        fetch_city_status("Tokyo")
    )

# Advanced Gradio Custom CSS for premium UI look with Center Alignment and Custom Spacing
custom_css = """
#app-wrap { max-width: 1200px; margin: 0 auto; padding: 30px; background: #ffffff; border-radius: 12px; }
.center-header { text-align: center; margin-bottom: 40px; }
.dashboard-card-custom { border-left: 6px solid #2563EB; padding: 18px; border-radius: 8px; margin-bottom: 15px; background: #F8FAFC; border-top: 1px solid #E2E8F0; border-right: 1px solid #E2E8F0; border-bottom: 1px solid #E2E8F0; }
.faq-card { background-color: #F8FAFC; border: 1px solid #E2E8F0; padding: 20px; border-radius: 10px; margin-top: 40px; }
.chatbot-container { border: 1px solid #E2E8F0; border-radius: 8px; padding: 10px; }
"""

with gr.Blocks(title="Nevin Weatherbot Hub") as demo:
    # Center-Aligned Logo and Header Setup
    with gr.Column(elem_id="app-wrap"):
        gr.HTML("""
        <div class='center-header' style='display: flex; flex-direction: column; align-items: center; justify-content: center; gap: 12px;'>
            <svg width="70" height="70" viewBox="0 0 64 64" fill="none" xmlns="http://www.w3.org/2000/svg" style='filter: drop-shadow(0px 4px 6px rgba(0, 0, 0, 0.1));'>
                <circle cx="24" cy="24" r="14" fill="#FDBA74"/>
                <path d="M46 44C46 38.48 41.52 34 36 34C34.9 34 33.84 34.15 32.84 34.42C30.6 30.56 26.46 28 21.6 28C14.64 28 9 33.64 9 40.6C9 41.08 9.03 41.56 9.08 42.03C5.54 43.14 3 46.42 3 50.3C3 55.1 6.9 59 11.7 59H45.3C50.1 59 54 55.1 54 50.3C54 46.82 50.62 44.5 46 44Z" fill="#94A3B8"/>
            </svg>
            <h1 style='color: #0F172A; margin: 0; font-size: 2.5rem; font-weight: 800; letter-spacing: -0.025em;'>NEVIN WEATHERBOT</h1>
            <p style='color: #64748B; margin: 0; font-size: 1.1rem;'>Professional Real-time Weather Dashboard & LLM Companion</p>
        </div>
        """)

        # Two-Column Layout Setup
        with gr.Row(equal_height=True):

            # Left Column: Interactive Live Dashboard
            with gr.Column(scale=1):
                gr.Markdown("### 🌍 Live World Capital Dashboard")

                with gr.Group(elem_classes=["dashboard-card-custom"]):
                    gr.Markdown("**🏙️ Dubai, UAE**")
                    dubai_widget = gr.Markdown("Loading temperature...")

                with gr.Group(elem_classes=["dashboard-card-custom"]):
                    gr.Markdown("**🇬🇧 London, United Kingdom**")
                    london_widget = gr.Markdown("Loading temperature...")

                with gr.Group(elem_classes=["dashboard-card-custom"]):
                    gr.Markdown("**🇺🇸 New York, USA**")
                    ny_widget = gr.Markdown("Loading temperature...")

                with gr.Group(elem_classes=["dashboard-card-custom"]):
                    gr.Markdown("**🇯🇵 Tokyo, Japan**")
                    tokyo_widget = gr.Markdown("Loading temperature...")

                refresh_btn = gr.Button("🔄 Refresh Dashboard Data", variant="secondary")

            # Right Column: Conversational AI Tab
            with gr.Column(scale=1):
                gr.Markdown("### 💬 Assistant Chat")

                # Main interactive chat interface with Niven Weatherbot
                chatbot_input = gr.Textbox(
                    label="Ask Niven Weatherbot (e.g., Should I carry an umbrella in London?)",
                    placeholder="Type your query here...",
                    lines=2
                )

                chatbot_output = gr.Textbox(
                    label="Response",
                    placeholder="Niven Weatherbot's response will appear here.",
                    interactive=False,
                    lines=8
                )

                submit_btn = gr.Button("🚀 Ask Weatherbot", variant="primary")

        # FAQ Component at bottom
        with gr.Group(elem_classes=["faq-card"]):
            gr.HTML("""
            <h4 style='margin-top: 0; color: #1E293B;'>💡 Frequently Asked Questions (FAQs)</h4>
            <div style='display: flex; flex-direction: column; gap: 12px; font-size: 0.95rem; color: #475569;'>
                <div><strong>Q: Where does the real-time weather information come from?</strong><br/>A: It links directly to OpenWeatherMap API using a preconfigured key.</div>
                <div><strong>Q: What happens if I request non-weather information?</strong><br/>A: The Niven Weatherbot model is specialized in weather and will politely redirect you back to weather queries.</div>
            </div>
            """)

    # Dynamic Wire actions
    submit_btn.click(fn=weather_chat, inputs=chatbot_input, outputs=chatbot_output)
    refresh_btn.click(fn=get_all_dashboard_cities, outputs=[dubai_widget, london_widget, ny_widget, tokyo_widget])

    # Trigger initial values loading upon application launch
    demo.load(fn=get_all_dashboard_cities, outputs=[dubai_widget, london_widget, ny_widget, tokyo_widget])

demo.launch(share=True, css=custom_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76585249db101c853b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
